# Exp7.3.8 — LIF current-gain sweep

Analysis-only notebook. Training and gain selection are performed by the Slurm array jobs; this notebook reads finalized artifacts only.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
artifact = root / 'notebooks/artifacts/experiment_7_3_8_lif_gain_sweep/lif_gain_sweep_v1'
method_summary = pd.read_csv(artifact / 'method_summary.csv')
gain_summary = pd.read_csv(artifact / 'gain_sweep_summary.csv')
contrast_summary = pd.read_csv(artifact / 'contrast_summary.csv')
bias_summary = pd.read_csv(artifact / 'count_bias_summary.csv')


## Method-level comparison
Only aggregated methods are shown here; per-seed rows remain in the finalized CSV artifacts.

In [ ]:
display(method_summary[['method', 'test_ba_mean', 'test_ba_std', 'selected_gain_mean']])
display(contrast_summary)


## Gain-response curves
Validation selects the gain; test curves are plotted only for post-hoc interpretation.

In [ ]:
for split in ['val', 'test']:
    fig, ax = plt.subplots(figsize=(7, 4))
    view = gain_summary[gain_summary['split'] == split]
    for source, group in view.groupby('weight_source'):
        group = group.sort_values('gain')
        ax.plot(group['gain'], group['balanced_accuracy_mean'], marker='o', label=source)
    ax.set_xscale('log', base=2)
    ax.set_xlabel('LIF current gain g')
    ax.set_ylabel(f'{split} balanced accuracy')
    ax.set_title(f'Exp7.3.8 {split} gain response')
    ax.legend()
    plt.show()


## Operating-regime diagnostics

In [ ]:
cols = ['weight_source', 'gain', 'split', 'zero_output_fraction_mean', 'mean_total_output_spikes_mean', 'mean_active_output_classes_mean', 'mean_per_class_count_variance_mean']
display(gain_summary[gain_summary['split'] == 'val'][cols].sort_values(['weight_source', 'gain']))


## Learned count-space offsets after gain selection

In [ ]:
display(bias_summary)
